### ממ"ן 11 - מימוש שידור (Broadcasting) ב-PyTorch
#### מגיש - Ohad Wolfman 316552496

<div dir="rtl">


- **א.** פונקציה שמקבלת טנזורים A ו-B ומחזירה את A משודר לממדי B (מימוש של `A.expand_as(B)`). אם השידור לא אפשרי, לזרוק שגיאה. אסור לשנות את B.

**אסור:** `expand`, `expand_as`, `repeat`, `broadcast_tensors`, `broadcast_to`, `vmap`, ואסור גם לעקוף דרך numpy.

**מותר:** `squeeze`, `unsqueeze`, `cat`, `stack`, `reshape`, `reshape_as`, `clone`.

</div>

In [2]:
import torch as t

In [ ]:
def my_expand_as(A, B):
    a_shape, b_shape = A.shape, B.shape

    # שלב 1: בדיקת תקינות
    if len(a_shape) > len(b_shape):
        raise ValueError(f"A has more dims than B: {tuple(a_shape)} vs {tuple(b_shape)}")
    
    for i in range(1, len(a_shape) + 1):    # ביצוע איטרציות מהסוף להתחלה
        a, b = a_shape[-i], b_shape[-i]
        if a != b and a != 1:   # אם שונים - אם לפחות איי הוא 1 אז כופלים את העמודה/המימדים בכמות המקבילה בבי
            raise ValueError(f"Cannot expand {tuple(a_shape)} to {tuple(b_shape)}: dim -{i} is {a} vs {b}")

    # שלב 2: השוואת מספר הממדים
    C = A.clone()   # הערך להחזרה עליו נבצע את ההרחבה
    while C.dim() < B.dim():
        C = C.unsqueeze(dim = 0)  # מוסיפים מימדים משמאל להשוות לגודל היעד - בי

    # שלב 3: מתיחה של כל מימד בגודל 1 לגדול המתאים
    for d in range(B.dim()):
        target = b_shape[d]     # הגודל הרצוי של המימד הנוכחי
        if C.shape[d] == 1 and target != 1:     # נבצע רק כאשר יש לאן להרחיב את המימד בסי
            if target == 0:     # הפונ' חיבור צריכה לפחות טנזור אחד, אחרת שגיאה
                C = C.narrow(dim=d, start=0, length=0)  # מתוך מימד די, קח 0 איברים החל מאינדקס 0
            else:
                C = t.cat([C] * target, dim=d)     # מדביקה את כמות העותקים זה לזה לאורך מימד די
    return C

In [ ]:
# בדיקה
A = t.rand((3,1))
B = t.zeros((1,3,2))  

my_expand_as(A, B)  

tensor([[[0.3658, 0.3658],
         [0.1321, 0.1321],
         [0.3805, 0.3805]]])

<div dir="rtl">

- **ב.** פונקציה שבודקת אם שני טנזורים ניתנים לשידור יחד, ומחזירה תמיד tuple מהצורה `(True/False, shape)`. ה-shape רלוונטי רק כשהערך הראשון הוא True.
</div>

In [ ]:
# בשידור דו כיווני כל טנזור יכול להימתח בממד אחר, ואילו בחד כיווני מוגדר שטנזור אחד הוא היעד הקבוע
def my_broadcast(A,B):
    a_shape, b_shape = A.shape, B.shape

    # שלב 1: בדיקת תקינות
    if len(a_shape) > len(b_shape):
        raise ValueError(f"A has more dims than B: {tuple(a_shape)} vs {tuple(b_shape)}")
    
    for i in range(1, len(a_shape) + 1):    # ביצוע איטרציות מהסוף להתחלה
        a, b = a_shape[-i], b_shape[-i]
        if a != b and a != 1:   # אם שונים - אם לפחות איי הוא 1 אז כופלים את העמודה/המימדים בכמות המקבילה בבי
            raise ValueError(f"Cannot expand {tuple(a_shape)} to {tuple(b_shape)}: dim -{i} is {a} vs {b}")

    # שלב 2: השוואת מספר הממדים
    C = A.clone()   # הערך להחזרה עליו נבצע את ההרחבה
    while C.dim() < B.dim():
        C = C.unsqueeze(dim = 0)  # מוסיפים מימדים משמאל להשוות לגודל היעד - בי

    # שלב 3: מתיחה של כל מימד בגודל 1 לגדול המתאים
    for d in range(B.dim()):
        target = b_shape[d]     # הגודל הרצוי של המימד הנוכחי
        if C.shape[d] == 1 and target != 1:     # נבצע רק כאשר יש לאן להרחיב את המימד בסי
            if target == 0:     # הפונ' חיבור צריכה לפחות טנזור אחד, אחרת שגיאה
                C = C.narrow(dim=d, start=0, length=0)  # מתוך מימד די, קח 0 איברים החל מאינדקס 0
            else:
                C = t.cat([C] * target, dim=d)     # מדביקה את כמות העותקים זה לזה לאורך מימד די
    return C


<div dir="rtl">

- **ג.** פונקציה שמשדרת שני טנזורים יחד ומחזירה שני טנזורים חדשים (מימוש של `torch.broadcast_tensors`). מותר להשתמש בסעיפים הקודמים.
</div>

<div dir="rtl">

- **ד.** הרצה על דוגמאות מגוונות והשוואה לפונקציות המובנות, כדי לוודא שכל חוקי השידור מומשו נכון.
</div>